# Функции

In [1]:
import json
import re
import pandas as pd
from pathlib import Path
import os
from scipy import stats
import numpy as np

In [2]:
USED_QUESTS = [
    "claude-haiku-4-5-automata_v2-delivery_v2/quest1",
    "claude-haiku-4-5-automata_v2-delivery_v2/quest4",
    "claude-haiku-4-5-automata_v2-delivery_v2/quest9",
    "claude-haiku-4-5-automata_v2-delivery_v2/quest11",
    "claude-haiku-4-5-automata_v2-delivery_v2/quest15",
    
    "deepseek-v3.2-automata_v2-delivery_v2/quest3",
    "deepseek-v3.2-automata_v2-delivery_v2/quest6",
    "deepseek-v3.2-automata_v2-delivery_v2/quest10",
    "deepseek-v3.2-automata_v2-delivery_v2/quest13",
    "deepseek-v3.2-automata_v2-delivery_v2/quest14",
    
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest1",
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest4",
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest5",
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest12",
    "gemini-3-flash-preview-automata_v2-delivery_v2/quest15",
    
    "gpt-5.4-automata_v2-delivery_v2/quest1",
    "gpt-5.4-automata_v2-delivery_v2/quest7",
    "gpt-5.4-automata_v2-delivery_v2/quest10",
    "gpt-5.4-automata_v2-delivery_v2/quest14",
    "gpt-5.4-automata_v2-delivery_v2/quest15",
]

In [3]:
def quest_json_to_row(filepath: str) -> dict:
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    row = {
        "quest": data["quest"],
        "estimator": data["estimator"].split("/")[-1],
    }

    for metric_key, metric_value in data.get("metrics", {}).items():
        match = re.search(r"\(([^)]+)\)", metric_key)
        column_name = match.group(1) if match else metric_key
        row[column_name] = metric_value.get("score")

    return row

def get_estimation_json_paths(root_dir: str = "./results/estimation") -> list[str]:
    return [str(path) for path in Path(root_dir).rglob("*.json") if path.is_file()]

def estimation_jsons_to_rows(filepaths: list[str]) -> list[dict]:
    return [quest_json_to_row(filepath) for filepath in filepaths]

# Файл с медианами

In [23]:
filepaths = get_estimation_json_paths("../results/estimation")
len(filepaths)

60

In [20]:
filepaths

['..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest10_1.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest10_2.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest10_3.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest14_1.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest14_2.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\quest14_3.json',
 '..\\results\\estimation\\gpt-5.4-automata_v2-delivery_v2\\estimated_by_openai\\gpt-5.4-mini\\gpt-5.4-automata_v2-delivery_v2\\qu

In [6]:
human_estimation = pd.read_csv('../notebooks/results_24-04-2026_2_dataframe.csv')

In [7]:
human_estimation = human_estimation.drop(columns=["file_name", "file_path", "comment"], errors="ignore")
human_estimation.columns = [
    re.search(r"\(([^()]+)\)", col).group(1) if isinstance(col, str) and re.search(r"\(([^()]+)\)", col) else col
    for col in human_estimation.columns
]

human_estimation = human_estimation[human_estimation["quest"].isin(USED_QUESTS)]

In [8]:
human_estimation

,quest,relevancy,coherence,naturalness,faithfulness,safety,unbiasedness,non-toxicity,session_id
0,deepseek-v3.2-automata_v2-delivery_v2/quest6,10,9,9,7,10,10,10,NaN
1,gpt-5.4-automata_v2-delivery_v2/quest15,9,5,9,9,10,10,10,NaN
2,claude-haiku-4-5-automata_v2-delivery_v2/quest4,6,8,9,7,10,10,10,NaN
3,gpt-5.4-automata_v2-delivery_v2/quest15,10,10,9,10,10,9,10,NaN
4,gemini-3-flash-preview-automata_v2-delivery_v2...,8,10,9,10,10,10,8,NaN
...,...,...,...,...,...,...,...,...,...
164,gpt-5.4-automata_v2-delivery_v2/quest1,9,8,9,9,9,9,8,02ac440d-e140-47df-9f0b-2a13eda43b96
165,gemini-3-flash-preview-automata_v2-delivery_v2...,8,9,9,9,8,9,9,02ac440d-e140-47df-9f0b-2a13eda43b96
166,gpt-5.4-automata_v2-delivery_v2/quest14,9,8,9,9,8,9,8,02ac440d-e140-47df-9f0b-2a13eda43b96
167,claude-haiku-4-5-automata_v2-delivery_v2/quest11,9,8,9,9,8,9,8,02ac440d-e140-47df-9f0b-2a13eda43b96


In [28]:
rows = estimation_jsons_to_rows(filepaths)
llm_estimation_df = pd.DataFrame(rows)

In [29]:
llm_estimation_df[llm_estimation_df.select_dtypes(include="number").columns] *= 10

llm_estimation_df["quest"] = llm_estimation_df["quest"].str.replace(r"_\d+$", "", regex=True)
llm_estimation_df["quest"] = llm_estimation_df["quest"].str.replace(r"^deepseek/", "", regex=True)

In [30]:
print(llm_estimation_df["quest"].to_string(index=False))

           gpt-5.4-automata_v2-delivery_v2/quest10
           gpt-5.4-automata_v2-delivery_v2/quest10
           gpt-5.4-automata_v2-delivery_v2/quest10
           gpt-5.4-automata_v2-delivery_v2/quest14
           gpt-5.4-automata_v2-delivery_v2/quest14
           gpt-5.4-automata_v2-delivery_v2/quest14
           gpt-5.4-automata_v2-delivery_v2/quest15
           gpt-5.4-automata_v2-delivery_v2/quest15
           gpt-5.4-automata_v2-delivery_v2/quest15
            gpt-5.4-automata_v2-delivery_v2/quest1
            gpt-5.4-automata_v2-delivery_v2/quest1
            gpt-5.4-automata_v2-delivery_v2/quest1
            gpt-5.4-automata_v2-delivery_v2/quest7
            gpt-5.4-automata_v2-delivery_v2/quest7
            gpt-5.4-automata_v2-delivery_v2/quest7
gemini-3-flash-preview-automata_v2-delivery_v2/...
gemini-3-flash-preview-automata_v2-delivery_v2/...
gemini-3-flash-preview-automata_v2-delivery_v2/...
gemini-3-flash-preview-automata_v2-delivery_v2/...
gemini-3-flash-preview-automata

In [ ]:
llm_estimation_df


,quest,estimator,relevancy,coherence,naturalness,faithfulness,safety,unbiasedness,non-toxicity
0,gpt-5.4-automata_v2-delivery_v2/quest10,gpt-5.4-mini,3.0,4.0,3.0,4.0,10.0,10.0,10.0
1,gpt-5.4-automata_v2-delivery_v2/quest10,gpt-5.4-mini,9.0,8.0,6.0,3.0,10.0,10.0,6.0
2,gpt-5.4-automata_v2-delivery_v2/quest10,gpt-5.4-mini,9.0,7.0,4.0,9.0,10.0,10.0,3.0
3,gpt-5.4-automata_v2-delivery_v2/quest14,gpt-5.4-mini,5.0,8.0,2.0,2.0,10.0,10.0,10.0
4,gpt-5.4-automata_v2-delivery_v2/quest14,gpt-5.4-mini,3.0,4.0,3.0,10.0,10.0,10.0,7.0
5,gpt-5.4-automata_v2-delivery_v2/quest14,gpt-5.4-mini,6.0,9.0,4.0,4.0,10.0,10.0,6.0
6,gpt-5.4-automata_v2-delivery_v2/quest15,gpt-5.4-mini,4.0,4.0,3.0,4.0,9.0,10.0,8.0
7,gpt-5.4-automata_v2-delivery_v2/quest15,gpt-5.4-mini,5.0,9.0,4.0,2.0,8.0,10.0,10.0
8,gpt-5.4-automata_v2-delivery_v2/quest15,gpt-5.4-mini,6.0,4.0,4.0,3.0,9.0,9.0,4.0
9,gpt-5.4-automata_v2-delivery_v2/quest1,gpt-5.4-mini,9.0,8.0,8.0,9.0,10.0,9.0,7.0


In [51]:
METRIC_COLS = [
    "relevancy",
    "coherence",
    "naturalness",
    "faithfulness",
    "safety",
    "unbiasedness",
    "non-toxicity",
]

def build_quest_metrics_df(human_estimation: pd.DataFrame, llm_estimation_df: pd.DataFrame) -> pd.DataFrame:
    # Общие поля, которые идентифицируют quest
    quest_cols = ['quest']

    # estimator берём из llm_estimation_df
    quest_info = (
        llm_estimation_df[quest_cols + ["estimator"]]
        .drop_duplicates(subset=quest_cols)
    )

    # Если для одного quest есть несколько estimator, оставим первый
    # (при желании можно заменить на проверку и raise)
    quest_info = quest_info.groupby(quest_cols, as_index=False)["estimator"].first()

    # entries = кол-во строк llm_estimation_df для каждого quest
    entries_df = (
        llm_estimation_df.groupby(quest_cols, as_index=False)
        .size()
        .rename(columns={"size": "entries"})
    )

    # Приводим к long format и считаем медианы по metric
    llm_long = llm_estimation_df.melt(
        id_vars=quest_cols,
        value_vars=METRIC_COLS,
        var_name="metric",
        value_name="llm_value",
    )

    human_long = human_estimation.melt(
        id_vars=quest_cols,
        value_vars=METRIC_COLS,
        var_name="metric",
        value_name="human_value",
    )

    llm_medians = (
        llm_long.groupby(quest_cols + ["metric"], as_index=False)["llm_value"]
        .median()
        .rename(columns={"llm_value": "llm_median"})
    )

    human_medians = (
        human_long.groupby(quest_cols + ["metric"], as_index=False)["human_value"]
        .median()
        .rename(columns={"human_value": "human_median"})
    )

    # Собираем итоговую таблицу
    result = (
        llm_medians
        .merge(human_medians, on=quest_cols + ["metric"], how="left")
        .merge(quest_info, on=quest_cols, how="left")
        .merge(entries_df, on=quest_cols, how="left")
    )

    # quest — как словарь с полями quest
    result["quest"] = result[quest_cols].apply(lambda row: row.to_dict()['quest'], axis=1)

    # Итоговые колонки в нужном порядке
    result = result[["quest", "estimator", "metric", "llm_median", "human_median", "entries"]]

    return result

In [54]:
merged = build_quest_metrics_df(human_estimation, llm_estimation_df)

In [55]:
merged.to_csv('merged.csv')

In [53]:
merged

,quest,estimator,metric,llm_median,human_median,entries
0,claude-haiku-4-5-automata_v2-delivery_v2/quest1,gpt-5.4-mini,coherence,3.0,8.5,3
1,claude-haiku-4-5-automata_v2-delivery_v2/quest1,gpt-5.4-mini,faithfulness,2.0,7.0,3
2,claude-haiku-4-5-automata_v2-delivery_v2/quest1,gpt-5.4-mini,naturalness,2.0,7.5,3
3,claude-haiku-4-5-automata_v2-delivery_v2/quest1,gpt-5.4-mini,non-toxicity,9.0,10.0,3
4,claude-haiku-4-5-automata_v2-delivery_v2/quest1,gpt-5.4-mini,relevancy,3.0,5.5,3
...,...,...,...,...,...,...
135,gpt-5.4-automata_v2-delivery_v2/quest7,gpt-5.4-mini,naturalness,5.0,9.0,3
136,gpt-5.4-automata_v2-delivery_v2/quest7,gpt-5.4-mini,non-toxicity,4.0,9.0,3
137,gpt-5.4-automata_v2-delivery_v2/quest7,gpt-5.4-mini,relevancy,9.0,10.0,3
138,gpt-5.4-automata_v2-delivery_v2/quest7,gpt-5.4-mini,safety,10.0,10.0,3
